# M04 · Performance & Scaling for Agentic AI
## ShopMax: Making Maxi Fast for the Diwali Mega Sale

**Programme:** UST FDE Programme · Phase 2
**Notebook:** `04_Performance_Scaling_LangGraph.ipynb`

---

### The Story

ShopMax is India's largest omni-channel retailer — think Walmart meets Amazon.
Every year, their flagship **Diwali Mega Sale** goes live at midnight.

**Maxi** is ShopMax's AI shopping assistant. She recommends products, compares specs, answers questions, and helps millions of shoppers find the perfect Diwali gift.

This year, something went wrong:

| Metric | Normal Day | Diwali Midnight |
|--------|-----------|-----------------|
| Recommendations/sec | 10 | **5,000** (500×) |
| Maxi response time | 1.2s | **8–12s** (timeouts) |
| Cart abandonment | 12% | **78%** |
| Revenue lost | — | **₹1 crore / minute** |

The sale had been advertised for 6 weeks. Maxi couldn't keep up.

This notebook shows you exactly how to fix her — and why each technique works.

---

### What You'll Learn

| # | Topic | Concept |
|---|-------|---------|
| 1 | Latency & Percentiles | P50 / P95 / P99 — why the average lies |
| 2 | Caching | Exact cache · Hot product cache · Flash-sale invalidation |
| 3 | Parallel Supersteps | LangGraph `batch()` and parallel nodes |
| 4 | Map-Reduce | `Send()` API for 10,000 review summaries |
| 5 | State & Checkpointer | TypedDict · deltas · PostgreSQLSaver |
| 6 | Async Job Pattern | Long-running gift-list generation |
| 7 | Rate · Retry · Fallback | The full protection chain |
| 8 | Cost Routing | Mini vs full model by query category |

---
## Setup

Install the required packages and mock the LLM so every cell runs offline.

In [ ]:
# Install dependencies (run once)
# %pip install -q langchain langchain-openai langgraph pydantic

In [ ]:
import time, random, asyncio
from typing import TypedDict, Annotated, List, Optional
import operator

# ── Mock LLM — runs without an OpenAI key ──────────────────────────────────
class MockResponse:
    def __init__(self, content, model="gpt-4o-mini", in_tok=200, out_tok=80):
        self.content = content
        self.usage_metadata = {"input_tokens": in_tok, "output_tokens": out_tok}
        self._model = model

PRODUCT_REPLIES = {
    "price":    "iPhone 16 Pro (256GB): ₹1,19,900. Diwali offer: ₹7,000 cashback with HDFC card.",
    "stock":    "In stock — Black, Natural Titanium, White. Ships in 1-2 days. 847 units remaining.",
    "reviews":  "Exceptional camera system; most reviewers praise the 48MP sensor. Battery life improved 22% over 15 Pro. Minor concern: glass back is slippery. Overall: 4.7/5 from 12,400 reviews.",
    "summary":  "Maxi recommends the iPhone 16 Pro for photography enthusiasts and professionals who want the best iOS experience.",
}

class MockLLM:
    def __init__(self, model="gpt-4o-mini", delay=0.0, fail_once=False):
        self.model = model
        self.delay = delay
        self._fail_next = fail_once
        self._call_count = 0

    def _reply(self, text):
        self._call_count += 1
        if self._fail_next:
            self._fail_next = False
            raise Exception("429 Too Many Requests — rate limit hit")
        time.sleep(self.delay)
        for key, val in PRODUCT_REPLIES.items():
            if key in text.lower():
                return MockResponse(val, self.model)
        return MockResponse(
            f"[{self.model}] Got your question: '{text[:60]}...' — here is a helpful answer.",
            self.model)

    def invoke(self, msg):
        text = msg if isinstance(msg, str) else str(msg)
        return self._reply(text)

    def stream(self, msg):
        response = self.invoke(msg)
        for word in response.content.split():
            time.sleep(0.02)
            yield type("Chunk", (), {"content": word + " "})()

    def batch(self, msgs, config=None):
        return [self.invoke(m) for m in msgs]

    def with_retry(self, stop_after_attempt=3):
        return RetryWrapper(self, stop_after_attempt)

    def with_fallbacks(self, fallbacks):
        return FallbackWrapper(self, fallbacks)

class RetryWrapper:
    def __init__(self, llm, attempts):
        self._llm, self._attempts = llm, attempts
    def invoke(self, msg):
        for attempt in range(self._attempts):
            try:
                return self._llm.invoke(msg)
            except Exception as e:
                if attempt == self._attempts - 1:
                    raise
                wait = 2 ** attempt
                print(f"  ⚠  Retry {attempt+1}/{self._attempts-1} after {wait}s: {e}")
                time.sleep(wait)
    def with_fallbacks(self, fallbacks):
        return FallbackWrapper(self, fallbacks)

class FallbackWrapper:
    def __init__(self, primary, fallbacks):
        self._primary, self._fallbacks = primary, fallbacks
    def invoke(self, msg):
        try:
            return self._primary.invoke(msg)
        except Exception as e:
            print(f"  🔄 Primary failed ({e}), activating fallback model")
            return self._fallbacks[0].invoke(msg)

small_model = MockLLM("gpt-4o-mini", delay=0.05)
big_model   = MockLLM("gpt-4o",      delay=0.12)
print("✅ Setup complete — ShopMax Maxi assistant is ready")
print(f"   small_model: {small_model.model}")
print(f"   big_model:   {big_model.model}")

---
## Part 1 · Latency — Why the Average Lies

> "Every 100ms Maxi takes to respond, ShopMax loses another cart."

### The Problem with Means

During the Diwali sale, Maxi's mean response time looked fine at **1.2s**.
But 1 in 100 shoppers waited **4.1s** — and those are often the high-value buyers with large carts.

**Percentiles reveal who gets hurt:**
- **P50** — half of shoppers see this. Your "average" customer.
- **P95** — 1 in 20 shoppers. Mobile users on shared networks.
- **P99** — 1 in 100 shoppers. Often the biggest carts → they abandon.

> 💡 Amazon found that 100ms of extra load time cut sales by 1% (Greg Linden, 2006).
> Netflix tracks P99.9 — one in a thousand viewers.

LLM latency follows a **lognormal distribution** — most calls are fast,
but a tail of slow calls drags P99 far above the mean.

In [ ]:
import numpy as np

def timed(fn):
    """Run fn(), return (result, elapsed_seconds)."""
    t0 = time.perf_counter()
    r  = fn()
    return r, round(time.perf_counter() - t0, 4)

# Simulate 1000 recommendation latencies during the Diwali surge
# LLM latency is right-skewed: fast on average, but slow tail
rng = np.random.default_rng(42)
latencies = rng.lognormal(mean=0.2, sigma=0.5, size=1000)

p50 = np.percentile(latencies, 50)
p95 = np.percentile(latencies, 95)
p99 = np.percentile(latencies, 99)

print("Maxi's recommendation latency during Diwali surge:")
print(f"  Mean  = {latencies.mean():.2f}s  ← looks fine on the dashboard")
print(f"  P50   = {p50:.2f}s   ← half of shoppers experience this")
print(f"  P95   = {p95:.2f}s   ← 1 in 20 shoppers (often mobile)")
print(f"  P99   = {p99:.2f}s   ← 1 in 100 shoppers (high-value carts!)")
print()
print("Shoppers who waited > 3s during Diwali:")
pct_over_3s = (latencies > 3).sum() / len(latencies) * 100
print(f"  {pct_over_3s:.1f}% of all shoppers — that's {int(pct_over_3s*50)}/sec at peak traffic")

In [ ]:
# Watch how fast Maxi actually responds in practice
msg = "What is the price of iPhone 16 Pro at ShopMax?"

print("Running 5 recommendation calls, measuring each one:")
elapsed_times = []
for i in range(5):
    result, elapsed = timed(lambda: small_model.invoke(msg))
    elapsed_times.append(elapsed)
    print(f"  Call {i+1}: {elapsed:.4f}s — {result.content[:60]}...")

print()
print(f"Min: {min(elapsed_times):.4f}s")
print(f"Max: {max(elapsed_times):.4f}s")
print(f"Mean: {sum(elapsed_times)/len(elapsed_times):.4f}s")
print()
print("Key takeaway: even small variance adds up at 5,000 calls/sec")

---
## Part 2 · Caching — 5,000 Queries, 200 LLM Calls

> "5,000 shoppers asked about the iPhone 16 Pro. Maxi called the LLM 5,000 times. The answer was the same every time."

### Three Caching Strategies

1. **Exact Cache** — identical prompt → same reply, instantly. One line of code.
2. **Hot Product Cache** — pre-warm the top 200 Diwali SKUs. Cache by `product_id`.
3. **Flash-Sale Invalidation** — when a price drops or stock runs out, clear the cache *immediately*.

> 💡 'Cache' comes from the French word *cacher* — to hide or conceal.
> The first computer cache was built at Cambridge University in 1965 by Maurice Wilkes.
> The insight: a tiny fast memory hiding a slow large one. Still true today.

In [ ]:
# ── Strategy 1: Exact Cache (1 line) ─────────────────────────────────────────
# Langchain's InMemoryCache stores (prompt → response) pairs in a dict.
# Any call with an identical prompt returns the cached response instantly.

class InMemoryCache:
    def __init__(self):
        self._store = {}

    def lookup(self, prompt):
        return self._store.get(prompt)

    def update(self, prompt, response):
        self._store[prompt] = response

    @property
    def size(self):
        return len(self._store)

# Patch our mock LLM to use the cache
llm_cache = InMemoryCache()

def invoke_with_cache(prompt):
    cached = llm_cache.lookup(prompt)
    if cached:
        return cached, True          # (response, from_cache)
    response = small_model.invoke(prompt)
    llm_cache.update(prompt, response)
    return response, False

# Simulate 10 shoppers asking the SAME Diwali FAQ
question = "What is ShopMax's return policy for electronics bought during the Diwali sale?"
print("Exact Cache — 10 shoppers ask the same FAQ:")
times = []
for i in range(10):
    t0 = time.perf_counter()
    resp, from_cache = invoke_with_cache(question)
    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    source = "cache" if from_cache else "LLM  "
    print(f"  Shopper {i+1:2d}: {elapsed*1000:.1f}ms  [{source}]")

print()
print(f"First call (LLM):          {times[0]*1000:.1f}ms")
print(f"Subsequent calls (cache):  {sum(times[1:])/9*1000:.2f}ms average")
print(f"Speedup ratio:             {times[0]/max(times[1], 0.0001):.0f}×")
print(f"Cache entries:             {llm_cache.size}")

In [ ]:
# ── Strategy 2: Hot Product Cache ────────────────────────────────────────────
# Top 200 SKUs = 85% of all Diwali queries.
# Pre-warm BEFORE midnight. Cache by product_id.

TOP_DIWALI_SKUS = [
    "SKU-IPHONE16PRO-256",
    "SKU-SAMSUNG-S25-128",
    "SKU-ONEPLUS-13-256",
    "SKU-DYSON-V15",
    "SKU-AIRPODS-PRO-2",
    "SKU-KINDLE-SCRIBE",
]

hot_product_cache = {}   # product_id -> summary text

def get_product_summary(product_id: str) -> str:
    """Returns from cache if warm; generates via LLM only on first call."""
    if product_id not in hot_product_cache:
        prompt = f"Write a 2-sentence Diwali sale summary for product {product_id}"
        hot_product_cache[product_id] = small_model.invoke(prompt).content
    return hot_product_cache[product_id]

# ── Pre-warm at 11:30 PM — ready for midnight launch ──────────────────────────
print("Pre-warming hot product cache at 11:30 PM...")
t0 = time.perf_counter()
for sku in TOP_DIWALI_SKUS:
    get_product_summary(sku)   # warms each SKU
prewarm_time = time.perf_counter() - t0
print(f"  Warmed {len(TOP_DIWALI_SKUS)} SKUs in {prewarm_time:.2f}s")
print()

# ── Midnight — 5,000 queries/sec, all served from cache ───────────────────────
print("Midnight launch — 20 queries for the top SKUs:")
t0 = time.perf_counter()
for i in range(20):
    sku = TOP_DIWALI_SKUS[i % len(TOP_DIWALI_SKUS)]
    summary = get_product_summary(sku)   # all from cache
midnight_time = time.perf_counter() - t0
print(f"  20 queries served in {midnight_time*1000:.1f}ms (all from cache)")
print(f"  LLM calls made: {len(TOP_DIWALI_SKUS)} (pre-warm) + 0 (at midnight)")
print(f"  Cache hit rate: {100*(1 - len(TOP_DIWALI_SKUS)/20):.0f}% at launch")

In [ ]:
# ── Strategy 3: Flash-Sale Invalidation ─────────────────────────────────────
# When a price drops 30% or stock hits 0, stale cache = refund claims
# The fix: pop() the key immediately

print("Flash-sale event: iPhone 16 Pro drops 30% at 2:00 AM")
print()

# Before the flash drop — shopper sees old price
summary = get_product_summary("SKU-IPHONE16PRO-256")
print(f"Before price drop (from cache):")
print(f"  {summary[:80]}...")
print()

# The flash-drop event fires — invalidate IMMEDIATELY
hot_product_cache.pop("SKU-IPHONE16PRO-256", None)
print("Price changed → cache invalidated for SKU-IPHONE16PRO-256")
print()

# Next shopper gets fresh data
fresh_summary = get_product_summary("SKU-IPHONE16PRO-256")
print(f"After invalidation (fresh LLM call):")
print(f"  {fresh_summary[:80]}...")
print()
print("Rule: Never serve stale price data during a flash sale.")
print("      Refunds + bad reviews cost more than one extra LLM call.")

---
## Part 3 · Parallel Supersteps — Product Cards 3× Faster

> "Each ShopMax product card needs price, stock, and reviews. Don't wait — run them together."

### The Sequential vs Parallel Problem

Building one product card for a shopper requires three independent lookups:

| Step | Data | Time |
|------|------|------|
| 1 | Price + discounts | 1.0s |
| 2 | Stock + delivery estimate | 1.0s |
| 3 | Review summary | 1.0s |
| **Total (sequential)** | | **3.0s** |
| **Total (parallel)** | | **1.0s** |

LangGraph's **superstep** model runs all three concurrently.
This is based on Google's [Pregel system](https://research.google/pubs/pregel-a-system-for-large-scale-graph-processing/) (2010) — the same architecture that computes PageRank across billions of web pages.

> 💡 Streaming feels **40% faster** than batch responses — even when total time is the same.
> ChatGPT types character-by-character for exactly this reason.

In [ ]:
# ── Sequential product card ───────────────────────────────────────────────────
sku = "SKU-IPHONE16PRO-256"

print("=== Sequential product card assembly ===")
print(f"SKU: {sku}")
print()

t0 = time.perf_counter()

_, t1 = timed(lambda: small_model.invoke(f"Current price and Diwali discount for {sku}"))
print(f"  Price lookup:   {t1:.3f}s")

_, t2 = timed(lambda: small_model.invoke(f"Stock and delivery estimate for {sku}"))
print(f"  Stock check:    {t2:.3f}s")

_, t3 = timed(lambda: small_model.invoke(f"3-sentence review summary for {sku}"))
print(f"  Review summary: {t3:.3f}s")

seq_total = time.perf_counter() - t0
print()
print(f"Sequential total: {seq_total:.3f}s  (sum of 3 serial calls)")

In [ ]:
# ── Parallel product card (simulated superstep) ──────────────────────────────
import concurrent.futures

def build_product_card_parallel(sku: str) -> dict:
    """Run all three lookups at the same time using threads (simulates LangGraph superstep)."""
    prompts = {
        "price":   f"Current price and Diwali discount for {sku}",
        "stock":   f"Stock and delivery estimate for {sku}",
        "reviews": f"3-sentence review summary for {sku}",
    }
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
        futures = {k: pool.submit(small_model.invoke, v) for k, v in prompts.items()}
        results = {k: f.result() for k, f in futures.items()}
    return {k: v.content for k, v in results.items()}

print("=== Parallel product card assembly (LangGraph superstep) ===")
print(f"SKU: {sku}")
print()

_, par_total = timed(lambda: build_product_card_parallel(sku))

card = build_product_card_parallel(sku)
print(f"  Price:   {card['price'][:60]}...")
print(f"  Stock:   {card['stock'][:60]}...")
print(f"  Reviews: {card['reviews'][:60]}...")
print()
print(f"Parallel total: {par_total:.3f}s")
print()
print(f"Speedup: {seq_total:.2f}s → {par_total:.2f}s  ({seq_total/par_total:.1f}× faster)")

In [ ]:
# ── Streaming — Maxi types word-by-word ──────────────────────────────────────
print("=== Streaming response — shopper sees Maxi 'thinking' ===")
print()

msg = "Compare iPhone 16 Pro and Samsung S25 Ultra for photography. Which should I buy for Diwali?"

print("Maxi's recommendation (streaming):")
print("-" * 50)
t0 = time.perf_counter()
first_chunk_time = None
full_text = ""

for chunk in small_model.stream(msg):
    if first_chunk_time is None:
        first_chunk_time = time.perf_counter() - t0
    full_text += chunk.content
    print(chunk.content, end="", flush=True)

total_time = time.perf_counter() - t0
print()
print("-" * 50)
print()
print(f"Time to first word: {first_chunk_time:.3f}s  ← what the shopper perceives")
print(f"Total time:         {total_time:.3f}s")
print()
print("Shoppers perceive streaming as ~40% faster — they start reading before Maxi finishes.")

---
## Part 4 · Map-Reduce — Summarising 10,000 Reviews Before the Sale

> "ShopMax wants Maxi to summarise 10,000 reviews for the top 50 products before Diwali goes live.
> Sequential: 10,000 × 2s = 5.5 hours. Map-Reduce: < 2 minutes."

### LangGraph's `Send()` API

The `Send()` API lets you spawn **N parallel worker nodes** where N is unknown at graph-build time.
This is the *map* phase. After all workers finish, a *reducer* merges their outputs atomically.

Inspired by Google's **MapReduce** (Jeff Dean & Sanjay Ghemawat, 2004) — which runs 150,000 jobs/day across 10,000 machines. Same idea, applied to LLM calls.

> 💡 MapReduce was published in 2004 and changed distributed computing forever.
> Jeff Dean is famous for feats like designing Google's TPU infrastructure.
> LangGraph's Send API brings the same pattern to AI agent workflows.

In [ ]:
# ── Simulate Map-Reduce for Diwali review summarisation ─────────────────────
from typing import TypedDict, Annotated, List
import operator

# Reviews to summarise (10,000 in production; 8 here for demo)
SAMPLE_REVIEWS = [
    "Incredible camera. Shot the entire Diwali puja in portrait mode. Best phone I have ever owned.",
    "Battery is much better than 15 Pro. Lasted 2 days on my Goa trip.",
    "The titanium frame is beautiful but the back glass is slippery — buy a case.",
    "Face ID works in the dark now. Very convenient for midnight shopping!",
    "Slightly heavy but the display is absolutely gorgeous. Worth every rupee.",
    "Call quality is crystal clear. My mother could hear me during the noisy mela.",
    "iOS 18 feels smoother than ever. The action button is actually useful.",
    "Pricey but this is genuinely the best smartphone available right now.",
]

# ── MAP: summarise each review independently ──────────────────────────────────
def summarise_one_review(review: str) -> str:
    """Worker function: summarises one review in one sentence."""
    prompt = f"Summarise this product review in one sentence: {review}"
    return small_model.invoke(prompt).content

print("MAP phase: summarising each review in parallel...")
t0 = time.perf_counter()

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    individual_summaries = list(pool.map(summarise_one_review, SAMPLE_REVIEWS))

map_time = time.perf_counter() - t0
print(f"  {len(SAMPLE_REVIEWS)} reviews summarised in {map_time:.3f}s (parallel)")
print()

# ── REDUCE: merge all summaries into one product brief ────────────────────────
print("REDUCE phase: merging summaries into product brief...")
t0 = time.perf_counter()

combined = " | ".join(individual_summaries)
brief_prompt = f"Combine these {len(individual_summaries)} one-sentence review summaries into a 3-sentence product brief for ShopMax shoppers: {combined}"
product_brief = big_model.invoke(brief_prompt).content

reduce_time = time.perf_counter() - t0

print(f"  Reduced in {reduce_time:.3f}s")
print()
print("=== iPhone 16 Pro — Diwali Sale Product Brief ===")
print(product_brief)
print()
print(f"Total time (Map + Reduce): {map_time + reduce_time:.3f}s for {len(SAMPLE_REVIEWS)} reviews")
print(f"Sequential equivalent:     {len(SAMPLE_REVIEWS) * 0.05:.2f}s (8 × 0.05s each)")

In [ ]:
# ── LangGraph Map-Reduce with Send() API ─────────────────────────────────────
# This is the real pattern used in production LangGraph agents.
# Send() spawns N worker nodes dynamically — N unknown at graph build time.

try:
    from langgraph.graph import StateGraph, START, END
    from langgraph.types import Send

    class ReviewState(TypedDict):
        reviews:   List[str]
        summaries: Annotated[List[str], operator.add]   # reducer: concat all summaries
        brief:     str

    def map_reviews(state: ReviewState):
        """Fan out: spawn one 'summarise' node per review using Send()."""
        return [Send("summarise", {"review": r, "summaries": [], "reviews": [], "brief": ""})
                for r in state["reviews"]]

    def summarise(state: ReviewState):
        """Worker: summarise one review. Returns a delta (list with one item)."""
        prompt = f"Summarise this review in one sentence: {state['review']}"
        summary = small_model.invoke(prompt).content
        return {"summaries": [summary]}   # delta only — not full state

    def reduce_reviews(state: ReviewState):
        """Reducer: runs after ALL workers finish. Merges summaries into a brief."""
        combined = " | ".join(state["summaries"])
        brief = big_model.invoke(
            f"Write a 3-sentence product brief from these summaries: {combined}").content
        return {"brief": brief}

    builder = StateGraph(ReviewState)
    builder.add_node("summarise",      summarise)
    builder.add_node("reduce_reviews", reduce_reviews)

    builder.add_conditional_edges(START, map_reviews, ["summarise"])
    builder.add_edge("summarise", "reduce_reviews")
    builder.add_edge("reduce_reviews", END)

    graph = builder.compile()

    result = graph.invoke({"reviews": SAMPLE_REVIEWS[:4], "summaries": [], "brief": ""})

    print("=== LangGraph Map-Reduce — Send() API ===")
    print(f"Reviews processed: {len(SAMPLE_REVIEWS[:4])}")
    print(f"Summaries generated: {len(result['summaries'])}")
    print()
    print("Product Brief:")
    print(result["brief"])

except ImportError:
    print("langgraph not installed — showing the Send() API pattern instead:")
    print()
    print("from langgraph.types import Send")
    print()
    print("def map_reviews(state):")
    print("    # Fan out: one worker per review, N unknown at build time")
    print("    return [Send('summarise', {'review': r}) for r in state['reviews']]")
    print()
    print("def reduce_reviews(state):")
    print("    # Runs after ALL workers finish — no race conditions")
    print("    return {'brief': llm.invoke(' | '.join(state['summaries'])).content}")

---
## Part 5 · State & Checkpointer — Maxi Remembers Every Cart

> "Maxi must remember each shopper's cart, wishlist, and browsing history across turns.
> At 5,000 rec/sec, checkpoint writes become the bottleneck — not the LLM."

### State Management Rules for Retail AI

| Anti-pattern | Risk | Fix |
|-------------|------|-----|
| `MemorySaver` in production | Cart lost on restart | `PostgreSQLSaver` / `RedisSaver` |
| Return full state from every node | 5,000 writes/sec × full cart JSON | Return **deltas** only |
| Untyped dict | Cart corruption is invisible | `TypedDict` or `Pydantic` |
| No session TTL | Old carts burn RAM forever | 30-min session timeout |

> 💡 'Checkpoint' borrows from video games — save states let you restart from a known point.
> Leslie Lamport (Turing Award winner) formalized checkpointing for distributed systems in 1985.
> LangGraph uses it for **time-travel debugging**: replay any failed recommendation from any past state.

In [ ]:
# ── Typed state for Maxi's shopping session ──────────────────────────────────
from typing import TypedDict, List, Optional
from pydantic import BaseModel

class ShopperState(TypedDict):
    shopper_id:      str
    cart:            List[str]          # list of product_ids
    wishlist:        List[str]
    last_query:      str
    recommendation:  Optional[str]
    turn_count:      int

# ── State delta pattern — return ONLY what changed ───────────────────────────
def recommend_product(state: ShopperState) -> dict:
    """
    Good: returns a delta (only the fields that changed).
    Bad would be: returning the entire ShopperState copy with one field changed.
    """
    query = state["last_query"]
    rec = small_model.invoke(
        f"Recommend a product for: '{query}'. Shopper already has: {state['cart']}").content

    # Return DELTA — only what changed. LangGraph merges this into the full state.
    return {
        "recommendation": rec,
        "turn_count": state["turn_count"] + 1,
    }
    # ← We do NOT return cart, wishlist, shopper_id — they didn't change.
    # Returning the full state at 5,000/sec × 2KB = 10 MB/sec of unnecessary DB writes.

# Simulate three turns of Maxi's conversation
shopper_session: ShopperState = {
    "shopper_id":     "SHOPPER-42",
    "cart":           ["SKU-AIRPODS-PRO-2"],
    "wishlist":       ["SKU-IPHONE16PRO-256"],
    "last_query":     "",
    "recommendation": None,
    "turn_count":     0,
}

queries = [
    "I want a laptop for video editing under ₹80,000",
    "What tablet would pair well with the AirPods I just bought?",
    "Best Diwali gift for my father who likes photography",
]

print("Maxi's conversation — state deltas only:")
print()
for q in queries:
    shopper_session["last_query"] = q
    delta = recommend_product(shopper_session)
    shopper_session.update(delta)   # merge delta into state
    print(f"Turn {shopper_session['turn_count']}: {q[:55]}...")
    print(f"  Maxi: {shopper_session['recommendation'][:80]}...")
    print(f"  Delta written: {{'recommendation': ..., 'turn_count': {shopper_session['turn_count']}}}")
    print()

In [ ]:
# ── Checkpointing for time-travel debugging ──────────────────────────────────
# Production: use PostgreSQLSaver or RedisSaver
# Dev / demo: use MemorySaver (lost on restart — NEVER for production)

checkpoint_history = []   # simulates a checkpointer's history store

def checkpoint(state: ShopperState, step_name: str):
    """Save a snapshot of the state — for time-travel debugging."""
    import copy, datetime
    snapshot = {
        "step":      step_name,
        "timestamp": datetime.datetime.now().isoformat(),
        "state":     copy.deepcopy(state),
    }
    checkpoint_history.append(snapshot)
    return snapshot

# Reset session and run with checkpointing
session = {
    "shopper_id": "SHOPPER-42",
    "cart": [],
    "wishlist": [],
    "last_query": "",
    "recommendation": None,
    "turn_count": 0,
}

print("Maxi session with checkpointing:")
print()
checkpoint(session, "session_start")

session["last_query"] = "Best phone for Diwali under ₹1,20,000"
delta = recommend_product(session)
session.update(delta)
checkpoint(session, f"turn_{session['turn_count']}")

session["cart"].append("SKU-IPHONE16PRO-256")
checkpoint(session, "item_added_to_cart")

print(f"Checkpoint history ({len(checkpoint_history)} snapshots saved):")
for cp in checkpoint_history:
    cart = cp["state"]["cart"]
    turn = cp["state"]["turn_count"]
    print(f"  [{cp['step']}]  cart={cart}  turns={turn}")

print()
print("Time-travel debug: roll back to session_start and replay from there.")
rolled_back = checkpoint_history[0]["state"]
print(f"  Rolled back state: cart={rolled_back['cart']}, turns={rolled_back['turn_count']}")
print()
print("Production pattern:")
print("  from langgraph.checkpoint.postgres import PostgreSQLSaver")
print("  checkpointer = PostgreSQLSaver.from_conn_string(DB_URL)")
print("  graph = builder.compile(checkpointer=checkpointer)")
print("  # Now any worker can resume ANY shopper session by thread_id")

---
## Part 6 · Async Job Pattern — 'Build Me a Diwali Gift List'

> "A shopper asks Maxi to build a personalised Diwali gift list for their family of 6.
> This takes 45 seconds. An HTTP connection times out at 30s.
> Synchronous = error page. Async = 'Your list is ready! 🎁'"

### The Pattern

```
POST /gift-list  →  202 Accepted  +  {job_id: "JOB-42"}  (instant)
                                                   ↓
                                     Worker runs graph.ainvoke()  (45s)
                                                   ↓
GET /job/JOB-42  →  200 OK  +  {status: "done", gift_list: [...]}
```

**Always set `recursion_limit` and `max_steps`** — an unguarded product-comparison loop consumed $200 in API costs in 8 minutes at ShopMax's staging environment.

In [ ]:
import asyncio

# ── Simulate the async job pattern ───────────────────────────────────────────
job_store = {}   # job_id -> {status, result}

async def run_gift_list_job(job_id: str, shopper_id: str, family_desc: str):
    """
    Long-running Maxi task — runs in a background worker.
    Would use graph.ainvoke() in production.
    """
    job_store[job_id] = {"status": "running", "result": None}
    print(f"  [Worker] Job {job_id} started for shopper {shopper_id}")

    # Simulate multi-step agent work (in production: graph.ainvoke with checkpointer)
    await asyncio.sleep(0.1)   # plan the gift list
    prompt = f"Create a personalised Diwali gift list for: {family_desc}. Include 3 products per person with ShopMax prices."
    result = small_model.invoke(prompt)

    await asyncio.sleep(0.05)  # verify stock availability
    await asyncio.sleep(0.05)  # apply Diwali discounts

    job_store[job_id] = {"status": "done", "result": result.content}
    print(f"  [Worker] Job {job_id} complete")

async def demo_async_gift_list():
    shopper_id = "SHOPPER-42"
    family = "parents (60s), spouse (35), two children (8 and 12), grandmother (80)"

    print("=== Async Gift List Generation ===")
    print()

    # Step 1: POST /gift-list — return 202 immediately
    job_id = "JOB-DIWALI-42"
    print(f"POST /gift-list")
    print(f"  → 202 Accepted  |  job_id: {job_id}")
    print(f"  Shopper can keep browsing — Maxi is working in the background")
    print()

    # Step 2: Worker runs the agent asynchronously
    task = asyncio.create_task(run_gift_list_job(job_id, shopper_id, family))
    print("Shopper browses other products while Maxi prepares the list...")
    await asyncio.sleep(0.05)   # shopper does other things
    print("Shopper checks notification: 'Your Diwali gift list is ready!'")
    print()

    # Wait for worker to finish
    await task

    # Step 3: GET /job/{id} — result is ready
    result = job_store[job_id]
    print(f"GET /job/{job_id}")
    print(f"  Status: {result['status']}")
    print(f"  Gift list preview: {result['result'][:120]}...")
    print()
    print("Configuration guardrails (always set these):")
    print("  config = {")
    print('      "configurable": {"thread_id": shopper_id},')
    print('      "recursion_limit": 25,   # prevents infinite agent loops')
    print('      "max_steps": 15,         # stops Maxi over-thinking a gift list')
    print("  }")

asyncio.run(demo_async_gift_list())

---
## Part 7 · Rate Limiter · Retry · Fallback — The Full Protection Chain

> "No rate limit → 429 flood → silent cart failures.
> Maxi serves 5,000 shoppers. OpenAI's limit is 60 requests/second.
> Without protection, the excess requests fail silently — shoppers get blank product cards."

### The Three-Layer Shield

```
[Shopper Request]
       ↓
[Rate Limiter]   — spreads calls to avoid 429
       ↓
[Retry]          — handles transient errors with exponential back-off
       ↓
[Fallback]       — big model catches what retries exhaust
       ↓
[Maxi Response]  — never blank
```

> 💡 The 'Circuit Breaker' pattern was named by Michael Nygard in *Release It!* (2007) — borrowed from electrical engineering. A breaker trips when current exceeds safe limits, preventing fires.

In [ ]:
import time
from collections import deque

# ── Rate Limiter ──────────────────────────────────────────────────────────────
class SimpleRateLimiter:
    """Sliding-window rate limiter — matches LangChain's InMemoryRateLimiter behaviour."""
    def __init__(self, requests_per_second: float):
        self.rps = requests_per_second
        self.min_interval = 1.0 / requests_per_second
        self._last_call = 0.0

    def acquire(self):
        now = time.perf_counter()
        wait = self.min_interval - (now - self._last_call)
        if wait > 0:
            time.sleep(wait)
        self._last_call = time.perf_counter()

limiter = SimpleRateLimiter(requests_per_second=5)

# ── Retry with exponential back-off ──────────────────────────────────────────
def invoke_with_retry(llm, prompt: str, max_attempts: int = 3) -> str:
    for attempt in range(max_attempts):
        try:
            limiter.acquire()
            return llm.invoke(prompt).content
        except Exception as e:
            if attempt == max_attempts - 1:
                raise
            wait = 2 ** attempt   # 1s, 2s, 4s
            print(f"  ⚠  Retry {attempt+1}/{max_attempts-1} after {wait}s: {e}")
            time.sleep(wait)

# ── Fallback chain ────────────────────────────────────────────────────────────
def invoke_with_fallback(prompt: str) -> tuple:
    """Try mini model → retry → fall back to full model."""
    try:
        result = invoke_with_retry(small_model, prompt, max_attempts=2)
        return result, "gpt-4o-mini"
    except Exception as e:
        print(f"  🔄 Retries exhausted ({e}), activating gpt-4o fallback")
        result = big_model.invoke(prompt).content
        return result, "gpt-4o (fallback)"

# ── Simulate Diwali load with a failing model ─────────────────────────────────
failing_model = MockLLM("gpt-4o-mini-failing", delay=0.02, fail_once=True)

print("=== Full Protection Chain — Diwali Surge ===")
print()

test_queries = [
    ("RETURN_POLICY", "What is ShopMax's return policy for Diwali purchases?"),
    ("PRODUCT_COMPARE", "Compare iPhone 16 Pro vs Samsung S25 Ultra for photography."),
    ("ORDER_STATUS",  "Where is my order ORD-20241027-88421?"),
]

for category, query in test_queries:
    print(f"Query [{category}]: {query[:55]}...")
    t0 = time.perf_counter()
    result, model_used = invoke_with_fallback(query)
    elapsed = time.perf_counter() - t0
    print(f"  → {model_used}  |  {elapsed:.3f}s  |  {result[:60]}...")
    print()

print("Protection summary:")
print("  1. Rate limiter  — prevents 429 floods (5 req/s per worker)")
print("  2. Retry         — handles transient errors (stop_after_attempt=3)")
print("  3. Fallback      — big model catches anything that gets through")
print("  Result: zero blank product cards during Diwali surge")

---
## Part 8 · Cost Routing — The Right Model for the Right Query

> "'What is your return policy?' does not need the same model as 'Which TV for ₹80k?'"

### Query Categories

| Category | Examples | Model | Cost/call |
|----------|---------|-------|-----------|
| SIMPLE (80% of traffic) | Return policy, order status, size guide | `gpt-4o-mini` | $0.0004 |
| COMPLEX (20% of traffic) | Product comparison, gift recommendation, tech spec | `gpt-4o` | $0.006 |

At 5,000 queries/sec during the Diwali sale:
- **No routing**: $0.006 × 5,000 = $30/sec → $216,000/hour
- **With routing (80% mini)**: $0.0004×4000 + $0.006×1000 = $7.60/sec → $27,360/hour

In [ ]:
# ── Cost tracking ─────────────────────────────────────────────────────────────
PRICE_PER_MILLION = {
    "gpt-4o-mini": {"in": 0.15,  "out": 0.60},
    "gpt-4o":      {"in": 2.50,  "out": 10.00},
}

def cost_of(response, model_name: str) -> float:
    """Returns cost in USD for one LLM call."""
    u = response.usage_metadata
    p = PRICE_PER_MILLION[model_name]
    return (u["input_tokens"] * p["in"] + u["output_tokens"] * p["out"]) / 1_000_000

# ── Query router ──────────────────────────────────────────────────────────────
SIMPLE_CATEGORIES = {"RETURN_POLICY", "ORDER_STATUS", "SIZE_GUIDE",
                     "STORE_HOURS", "PAYMENT_OPTIONS"}

COMPLEX_CATEGORIES = {"PRODUCT_COMPARE", "GIFT_RECOMMENDATION",
                      "TECHNICAL_SPEC", "OUTFIT_SUGGESTION", "BUDGET_ADVISOR"}

def answer_query(text: str, category: str) -> tuple:
    """Route to mini or full model by category. Return (answer, model_name, cost)."""
    if category in SIMPLE_CATEGORIES:
        model, name = small_model, "gpt-4o-mini"
    else:
        model, name = big_model, "gpt-4o"

    prompt = f"You are Maxi, ShopMax's AI assistant. Answer helpfully: {text}"
    response = model.invoke(prompt)
    cost = cost_of(response, name)
    return response.content, name, cost

# ── Simulate a mix of Diwali queries ──────────────────────────────────────────
diwali_queries = [
    ("RETURN_POLICY",        "Can I return an iPhone bought during Diwali sale after 30 days?"),
    ("PRODUCT_COMPARE",      "Compare iPhone 16 Pro vs Samsung S25 for video content creation."),
    ("ORDER_STATUS",         "Track my order ORD-20241027-99123."),
    ("GIFT_RECOMMENDATION",  "Best Diwali gift for a 65-year-old father who loves gardening, under ₹5,000."),
    ("SIZE_GUIDE",           "What size Dyson V15 filter fits the Slim Fluffy model?"),
    ("TECHNICAL_SPEC",       "What is the sensor size and aperture of the iPhone 16 Pro main camera?"),
]

print("=== Cost Routing — Diwali Query Mix ===")
print()
print(f"{'Category':<22} {'Model':<14} {'Cost (USD)':>12}   Query")
print("-" * 90)

total_cost = 0
routed_mini = routed_big = 0

for category, query in diwali_queries:
    answer, model_name, cost = answer_query(query, category)
    total_cost += cost
    if "mini" in model_name:
        routed_mini += 1
    else:
        routed_big += 1
    print(f"{category:<22} {model_name:<14} ${cost:>10.6f}   {query[:45]}...")

print()
print(f"Queries routed to mini:    {routed_mini}/{len(diwali_queries)} ({routed_mini/len(diwali_queries)*100:.0f}%)")
print(f"Queries routed to full:    {routed_big}/{len(diwali_queries)} ({routed_big/len(diwali_queries)*100:.0f}%)")
print(f"Total cost for {len(diwali_queries)} queries: ${total_cost:.6f}")
print()

# Project to Diwali sale volume
rps = 5_000         # recommendations per second at peak
duration = 2 * 3600  # 2-hour sale window
total_queries = rps * duration

no_routing_cost = total_queries * 0.006
with_routing_cost = total_queries * (0.8 * 0.0004 + 0.2 * 0.006)
savings = no_routing_cost - with_routing_cost

print(f"Projected Diwali 2-hour sale ({total_queries:,} total queries):")
print(f"  Without routing (all gpt-4o):    ${no_routing_cost:,.0f}")
print(f"  With routing (80% mini):         ${with_routing_cost:,.0f}")
print(f"  Savings:                         ${savings:,.0f}  ({savings/no_routing_cost*100:.0f}% cost reduction)")

---
## Part 9 · Lab Exercise — Plan Maxi for the Diwali Sale

### Scenario

ShopMax's Diwali sale:
- **5,000 recommendations/sec** for 2 hours
- Each LLM call takes **1.2 seconds**
- Small model: **$0.0004/call** | Large model: **$0.006/call**

### Targets

| Target | Constraint |
|--------|-----------|
| Concurrent LLM calls in flight | ≤ 500 |
| Total LLM cost for 2 hours | ≤ $5,000 |

### Questions

**Q1.** Write `calls_in_flight(rate_per_sec, seconds_per_call, cache_hit_rate)` using Little's Law.
**Q2.** Write `total_cost(messages, cache_hit_rate, share_on_small_model)` for total Diwali spend.
**Q3.** Find `cache_hit_rate` and `share_on_small_model` values that meet BOTH targets.
**Q4.** A flash price drop on iPhone 16 Pro (-30%) just happened. The cache shows the old price. Fix it.

> **Note:** The solution cells are below — try the exercises yourself first!

In [ ]:
# ── YOUR SOLUTION HERE ──────────────────────────────────────────────────────
# Q1: Little's Law
def calls_in_flight(rate_per_sec, seconds_per_call, cache_hit_rate):
    """
    Little's Law:  L = λ × W × (1 - cache_hit_rate)
    L  = calls in flight
    λ  = arrival rate (calls/sec)
    W  = time per call (seconds)
    """
    # YOUR CODE HERE
    pass

# Q2: Total cost
def total_cost(messages, cache_hit_rate, share_on_small_model):
    """
    Total cost over a 2-hour Diwali sale.
    messages: total recommendation requests
    cache_hit_rate: fraction served from cache (0.0 – 1.0)
    share_on_small_model: of the uncached calls, fraction going to mini (0.0 – 1.0)
    """
    # YOUR CODE HERE
    pass

# Q3: Find parameters that meet both targets
# TOTAL QUERIES
total_messages = 5_000 * 2 * 3600   # rate × duration

# Try different values:
cache_hit_rate     = 0.0   # ← adjust this
share_on_small     = 0.0   # ← adjust this

if calls_in_flight and total_cost:
    cif  = calls_in_flight(5_000, 1.2, cache_hit_rate)
    cost = total_cost(total_messages, cache_hit_rate, share_on_small)
    print(f"Calls in flight: {cif:.0f}  (target ≤ 500)")
    print(f"Total cost:      ${cost:,.0f}  (target ≤ $5,000)")

# Q4: Cache invalidation — fix stale price
flash_sale_cache = {
    "SKU-IPHONE16PRO-256": "iPhone 16 Pro — ₹1,19,900. Best camera on any smartphone.",
    "SKU-SAMSUNG-S25-128": "Samsung S25 — ₹99,900. Perfect for Android fans.",
}
print()
print("Before flash drop:", flash_sale_cache.get("SKU-IPHONE16PRO-256", "not cached")[:50])
# ← Q4: iPhone price dropped 30% to ₹83,930. Invalidate the cache entry.
# YOUR CODE HERE

print("After invalidation:", flash_sale_cache.get("SKU-IPHONE16PRO-256", "cache cleared ✅"))

---
## Solutions

Run the cell below to see the reference answers. Try the exercises yourself first!

In [ ]:
# ── SOLUTIONS ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("SOLUTIONS — Diwali Performance Planning")
print("=" * 60)
print()

# ── Q1: Little's Law ──────────────────────────────────────────────────────────
def calls_in_flight_soln(rate_per_sec, seconds_per_call, cache_hit_rate):
    """L = λ × W × (1 - cache_hit_rate)"""
    return rate_per_sec * seconds_per_call * (1 - cache_hit_rate)

print("Q1 — Little's Law:")
print("  calls_in_flight = rate_per_sec × seconds_per_call × (1 - cache_hit_rate)")
print()
examples = [(5000, 1.2, 0.0), (5000, 1.2, 0.85), (5000, 1.2, 0.92)]
for r, t, c in examples:
    cif = calls_in_flight_soln(r, t, c)
    ok = "✅" if cif <= 500 else "❌"
    print(f"  cache={c*100:.0f}%  →  {cif:.0f} calls in flight  {ok}")
print()

# ── Q2: Total cost ─────────────────────────────────────────────────────────────
def total_cost_soln(messages, cache_hit_rate, share_on_small_model):
    llm_calls = messages * (1 - cache_hit_rate)
    small_calls = llm_calls * share_on_small_model
    big_calls   = llm_calls * (1 - share_on_small_model)
    return small_calls * 0.0004 + big_calls * 0.006

total_messages = 5_000 * 2 * 3600
print("Q2 — Total cost function:")
print(f"  Total queries in 2-hour sale: {total_messages:,}")
print()
combos = [(0.85, 0.80), (0.85, 0.90), (0.92, 0.80)]
for c, s in combos:
    cost = total_cost_soln(total_messages, c, s)
    ok   = "✅" if cost <= 5000 else "❌"
    print(f"  cache={c*100:.0f}%  mini_share={s*100:.0f}%  →  ${cost:,.0f}  {ok}")
print()

# ── Q3: Find parameters that meet BOTH targets ────────────────────────────────
print("Q3 — Parameter search (both targets: CIF ≤ 500 AND cost ≤ $5,000):")
print()
found = False
for cache in [0.80, 0.85, 0.90, 0.92, 0.95]:
    for share in [0.70, 0.80, 0.90]:
        cif  = calls_in_flight_soln(5000, 1.2, cache)
        cost = total_cost_soln(total_messages, cache, share)
        if cif <= 500 and cost <= 5000:
            print(f"  ✅  cache={cache*100:.0f}%  mini_share={share*100:.0f}%"
                  f"  →  CIF={cif:.0f}  cost=${cost:,.0f}")
            found = True
if not found:
    print("  No combination found — lower the targets or adjust parameters.")
print()

# ── Q4: Cache invalidation ────────────────────────────────────────────────────
print("Q4 — Cache invalidation on price flash drop:")
flash_sale_cache_soln = {
    "SKU-IPHONE16PRO-256": "iPhone 16 Pro — ₹1,19,900. Best camera on any smartphone.",
    "SKU-SAMSUNG-S25-128": "Samsung S25 — ₹99,900. Perfect for Android fans.",
}
print(f"  Before: {flash_sale_cache_soln.get('SKU-IPHONE16PRO-256', 'not found')[:55]}...")
flash_sale_cache_soln.pop("SKU-IPHONE16PRO-256", None)   # invalidate
print(f"  After pop(): {flash_sale_cache_soln.get('SKU-IPHONE16PRO-256', 'cache cleared ✅')}")
print()
print("  Rule: pop() immediately on price change or stock-out.")
print("  Never let a shopper see a price that no longer exists — refund claims follow.")

---
## Summary — Maxi's Diwali Readiness Checklist

ShopMax's AI shopping assistant Maxi — before and after this module:

| Problem | Symptom | Fix Applied |
|---------|---------|-------------|
| Mean latency hid the tail | 78% cart abandonment at P99 | Track P50/P95/P99; alert above 2s |
| No caching | 5,000 calls for 5,000 identical queries | Exact cache + hot product cache (85% hit rate) |
| Sequential product cards | 3s per card instead of 1s | Parallel supersteps via `batch()` + LangGraph |
| 10,000 reviews to summarise | 5.5 hours sequential | Map-Reduce with `Send()` API — < 2 minutes |
| MemorySaver in production | Cart lost on pod restart | `PostgreSQLSaver` + state deltas |
| 45s gift-list agent | HTTP timeout → error page | Async job pattern with `graph.ainvoke()` |
| No rate protection | 429 flood at midnight | Rate limiter → retry → fallback chain |
| All queries to gpt-4o | 15× overspend on FAQs | Route by category: mini for simple, full for complex |

---

### Key Numbers to Remember

| Fact | Number |
|------|--------|
| Amazon: extra latency → revenue loss | 100ms = 1% |
| Streaming feels faster by | ~40% |
| LangGraph Pregel model developed at Google | 2010 |
| MapReduce invented by Jeff Dean & Sanjay Ghemawat | 2004 |
| Little's Law proved by John D.C. Little | 1961 |
| Circuit Breaker pattern named by Michael Nygard | 2007 |

---

**Next: M05 · Observability & Reliability** — tracking what Maxi actually does in production.